# HDBSCAN (Hierarchical Density-Based Spatial Clustering)

Density-based clustering that finds clusters of varying density, automatically determines the number of clusters, and identifies noise points. Unlike KMeans, it does not assume spherical clusters and handles irregular shapes well.

**When to Use:**
- Unknown number of clusters
- Clusters of different shapes, sizes, and densities
- Noisy data with outliers (noise points labeled as -1)
- Spatial or geographic data
- When you need robust clustering without specifying k

**Key Assumptions / Considerations:**
- Clusters are defined by regions of high density separated by low density
- min_cluster_size is the most important parameter — minimum group size to form a cluster
- min_samples controls conservativeness — higher values = more points classified as noise
- Features should be scaled (density is distance-based)
- May struggle with very high-dimensional data — consider PCA first
- Noise label (-1) is a feature, not a bug — these are genuine outliers

**References:**
- [sklearn HDBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html)
- [HDBSCAN Paper](https://doi.org/10.1007/978-3-642-37456-2_14)
- [sklearn Clustering Comparison](https://scikit-learn.org/stable/modules/clustering.html)

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import HDBSCAN, KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# test data (with natural clusters + noise)

np.random.seed(100)
n_samples = 8000
n_noise = 400

# 3 clusters with different characteristics
cluster1_size = (n_samples - n_noise) // 3
cluster2_size = (n_samples - n_noise) // 3
cluster3_size = n_samples - n_noise - cluster1_size - cluster2_size

age = np.concatenate([
    np.random.normal(25, 4, cluster1_size),
    np.random.normal(45, 8, cluster2_size),
    np.random.normal(65, 4, cluster3_size),
    np.random.uniform(18, 70, n_noise)  # noise
]).astype(int)

income = np.concatenate([
    np.random.normal(30000, 4000, cluster1_size),
    np.random.normal(60000, 10000, cluster2_size),
    np.random.normal(90000, 6000, cluster3_size),
    np.random.uniform(10000, 120000, n_noise)  # noise
])

num_transactions = np.concatenate([
    np.random.poisson(5, cluster1_size),
    np.random.poisson(15, cluster2_size),
    np.random.poisson(25, cluster3_size),
    np.random.poisson(12, n_noise)  # noise
])

total = len(age)
gender = np.random.choice(['Male', 'Female'], total)
region = np.random.choice(['North', 'South', 'East', 'West'], total)
product_type = np.random.choice(['A', 'B', 'C'], total)

# ground truth (-1 for noise)
true_labels = np.concatenate([
    np.zeros(cluster1_size),
    np.ones(cluster2_size),
    np.full(cluster3_size, 2),
    np.full(n_noise, -1)
]).astype(int)

# shuffle
idx = np.random.permutation(total)
age = age[idx]
income = income[idx]
num_transactions = num_transactions[idx]
gender = gender[idx]
region = region[idx]
product_type = product_type[idx]
true_labels = true_labels[idx]

df = pd.DataFrame({
    'age': age,
    'income': income,
    'num_transactions': num_transactions,
    'gender': gender,
    'region': region,
    'product_type': product_type,
})

In [ ]:
# Load Data

#df = pd.read_csv("data.csv")

X = df.copy()

numeric_features = ['age', 'income', 'num_transactions']
categorical_features = ['gender', 'region', 'product_type']

In [ ]:
# Data Overview

print(df.shape)
display(df.describe())

fig, axes = plt.subplots(1, len(numeric_features), figsize=(5 * len(numeric_features), 4))
for i, col in enumerate(numeric_features):
    sns.histplot(df[col], kde=True, bins=30, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

In [ ]:
# Preprocessing

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()
print(f"\u2705 Processed features: {len(feature_names)}")

In [ ]:
# Parameter Exploration

min_cluster_sizes = [5, 10, 20, 50, 100, 200]
min_samples_list = [5, 10, 20]

param_results = []

for mcs in min_cluster_sizes:
    for ms in min_samples_list:
        hdb = HDBSCAN(min_cluster_size=mcs, min_samples=ms)
        labels = hdb.fit_predict(X_processed)
        
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        noise_pct = n_noise / len(labels) * 100
        
        sil = silhouette_score(X_processed, labels) if n_clusters > 1 else -1
        
        param_results.append({
            "min_cluster_size": mcs,
            "min_samples": ms,
            "n_clusters": n_clusters,
            "n_noise": n_noise,
            "noise_pct": round(noise_pct, 1),
            "silhouette": round(sil, 4)
        })

param_df = pd.DataFrame(param_results)
print("\u2705 Parameter Exploration Results:")
display(param_df)

In [ ]:
# Fit Final Model

# Select params with best silhouette (and reasonable noise %)
best_params = param_df[param_df['n_clusters'] >= 2].sort_values('silhouette', ascending=False).iloc[0]
best_mcs = int(best_params['min_cluster_size'])
best_ms = int(best_params['min_samples'])

print(f"\U0001f3c6 Selected: min_cluster_size={best_mcs}, min_samples={best_ms}")

hdbscan_final = HDBSCAN(min_cluster_size=best_mcs, min_samples=best_ms)
cluster_labels = hdbscan_final.fit_predict(X_processed)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = (cluster_labels == -1).sum()

print(f"   Clusters found: {n_clusters}")
print(f"   Noise points: {n_noise} ({n_noise/len(cluster_labels)*100:.1f}%)")

In [ ]:
# Cluster Evaluation

# Exclude noise for silhouette calculation
mask = cluster_labels != -1

print("\u2705 Cluster Evaluation Metrics (excluding noise):")
if mask.sum() > 0 and n_clusters > 1:
    print(f"   Silhouette Score:       {silhouette_score(X_processed[mask], cluster_labels[mask]):.4f}")
    print(f"   Calinski-Harabasz Score: {calinski_harabasz_score(X_processed[mask], cluster_labels[mask]):.4f}")
    print(f"   Davies-Bouldin Score:   {davies_bouldin_score(X_processed[mask], cluster_labels[mask]):.4f}")

print(f"\n   Cluster sizes:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for c, n in zip(unique, counts):
    label = "Noise" if c == -1 else f"Cluster {c}"
    print(f"   {label}: {n} samples ({n/len(cluster_labels)*100:.1f}%)")

In [ ]:
# Cluster Visualization (PCA)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# HDBSCAN clusters (noise in gray)
colors = cluster_labels.copy().astype(float)
noise_mask = cluster_labels == -1

axes[0].scatter(X_pca[~noise_mask, 0], X_pca[~noise_mask, 1], 
                c=cluster_labels[~noise_mask], cmap='viridis', alpha=0.5, s=10)
axes[0].scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], 
                c='gray', alpha=0.3, s=5, label=f"Noise ({noise_mask.sum()})")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
axes[0].set_title("HDBSCAN Clusters")
axes[0].legend()

# Ground truth
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=true_labels, cmap='viridis', alpha=0.5, s=10)
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
axes[1].set_title("Ground Truth Labels")
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Noise Analysis

df_analysis = df.copy()
df_analysis['cluster'] = cluster_labels
df_analysis['is_noise'] = cluster_labels == -1

print("\u2705 Noise vs Clustered - Numeric Feature Comparison:")
comparison = df_analysis.groupby('is_noise')[numeric_features].agg(['mean', 'std'])
display(comparison)

fig, axes = plt.subplots(1, len(numeric_features), figsize=(5 * len(numeric_features), 4))
for i, col in enumerate(numeric_features):
    sns.boxplot(data=df_analysis, x='is_noise', y=col, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xticklabels(['Clustered', 'Noise'])
plt.tight_layout()
plt.show()

In [ ]:
# Cluster Profiling

df_clustered = df_analysis[df_analysis['cluster'] != -1]

print("\u2705 Cluster Profiles (Numeric Features):")
profile = df_clustered.groupby('cluster')[numeric_features].mean()
display(profile)

scaler = StandardScaler()
profile_scaled = pd.DataFrame(
    scaler.fit_transform(profile),
    columns=numeric_features,
    index=profile.index
)

plt.figure(figsize=(10, 5))
sns.heatmap(profile_scaled.T, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Cluster Centers (Standardized)")
plt.xlabel("Cluster")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
# Comparison with KMeans

km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
km_labels = km.fit_predict(X_processed)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# HDBSCAN
axes[0].scatter(X_pca[~noise_mask, 0], X_pca[~noise_mask, 1], 
                c=cluster_labels[~noise_mask], cmap='viridis', alpha=0.5, s=10)
axes[0].scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], 
                c='gray', alpha=0.3, s=5)
axes[0].set_title(f"HDBSCAN ({n_clusters} clusters, {noise_mask.sum()} noise)")

# KMeans
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap='viridis', alpha=0.5, s=10)
axes[1].set_title(f"KMeans ({n_clusters} clusters, 0 noise)")

for ax in axes:
    ax.set_xlabel(f"PC1")
    ax.set_ylabel(f"PC2")

plt.tight_layout()
plt.show()

print(f"\U0001f4a1 HDBSCAN identifies {noise_mask.sum()} noise points that KMeans forces into clusters")

In [ ]:
# future work:
#   OPTICS as an alternative density-based algorithm
#   t-SNE / UMAP for better 2D visualization of high-dimensional clusters
#   soft clustering: use hdbscan library's prediction_data=True for probabilities
#   cluster stability analysis across different parameter settings
#   feature engineering to improve cluster separation
#   semi-supervised: use cluster labels as features for downstream models